## Overview 
This notebook analyzes EngageMetrics' approach to flexible work arrangements by investigating whether work modes (REMOTE, HYBRID, ONSITE) influence promotion eligibility. Through progressively independent analysis, for this task, employee_insights_cleanded dataset will be used to test relationships between work arrangements and career advancement opportunities.

To do so, the proposed flow is to form hypotheses, conduct statistical tests, and draw conclusions that could help EngageMetrics optimize their approach to remote work and promotion decisions.

## Learning Outcomes 
- Form and test statistical hypotheses about workplace relationships
- Conduct chi-square tests to analyze categorical relationships
- Interpret statistical test results in business context
- Visualize relationships between variables

## Dataset Information 
The <b>employee_insights_cleaned dataset</b> contains employee records including:
- work_mode (REMOTE, HYBRID, ONSITE)
- promotion_eligible (Y/N)
- department
- work_experience (years)
- satisfaction_score
- projects_completed

## Activities
### Activity 1: Hypothesis Formation and Initial Exploration

<b>Step 1:</b> Form Your Hypothesis. Before examining the data:
1. What factors might influence the relationship between work mode and promotions?
2. Write your hypothesis about this relationship.
3. Document your reasoning.

Discussion prompts:
- How might different work modes affect visibility for promotion?
- Could department or experience levels impact this relationship?
-What other factors should we consider?

<b>Tip:</b> Consider both direct and indirect factors that might influence promotion eligibility.

#### Factors that might influence promotions
- visibility: some modes may make work more visible than others, thus influencing elegibility. Sometimes looking busy affects more than being busy
- other variables like satisfaction_score might have an impact in elegibility. demotivated employees could be less likely to be promoted. satisfacion could also affect projects completed which could in turn affect elegibility for promotions
- experience: there could be a correlation between modes an experience. Experienced workers might prefer a specific mode of work. These workers might be more likely to be promoted than inexperienced ones

$$ 
 H_{0} \to work \ mode \ has \ no \ effect \ on \ promotions
$$

<br>
</br>

$$
 H_{1} \to promotions \ depend \ on \ work \ mode
$$

<b>Step 2:</b> Import necessary libraries and preview data

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

<b>Step 3:</b> Adjust paths to import data

In [ ]:
BASE_DIR = Path.cwd().resolve().parents[1]
DATA_DIR = BASE_DIR / "data"
employee_insights_path = DATA_DIR / "employee_insights_cleaned.csv"

In [ ]:
df = pd.read_csv(employee_insights_path)
df.head()

<b>Step 4:</b> Initial Data Exploration

In [ ]:
print('\nWork mode distribution')
print(df['work_mode'].value_counts())

print('\nPromotions distribution')
print(df['promotion_eligible'].value_counts())

In [ ]:
# Basic relationship visualization
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='work_mode', hue='promotion_eligible')
plt.title('Promotion Eligibility by Work Mode')
plt.show()

### Activity 2: Statistical Analysis

<b>Step 1:</b> Create a contingency table examining work_mode and promotion_eligible

Questions to consider:
- What patterns do you notice in the contingency table?
- Do these patterns align with your hypothesis?

In [ ]:
contingency_table = pd.crosstab(df['work_mode'], df['promotion_eligible'])
contingency_table_per = pd.crosstab(df['work_mode'], df['promotion_eligible'], normalize='index')

In [ ]:
sns.heatmap(contingency_table, annot=True, fmt='d', cmap='PuBu')
plt.title('Work mode elegibility patterns for promotions')

In [ ]:
display(contingency_table_per)

The pattern is opposite to the expected, there is difference in elegibility in favor of remote workers. 

<b>Step 2:</b> Conduct Perform Chi-Square Test to examine the relationship.

Questions:
- What does the p-value tell us?
- How strong is the relationship?

In [ ]:
stat, p, dof, expected = chi2_contingency(contingency_table)
print(f'p-value: {p:.4f}')

### Activity 3: Advanced Analysis 

<b>Step 1:</b> Investigate Confounding Factors. Analyze how other factors might influence the relationship:
1. Examine promotion rates by department within each work mode
2. Consider work experience as a factor
3. Visualize any patterns you find

#### Analysis by department and mode

In [ ]:
dept_work_contingency = pd.crosstab(
    [df['department'], df['work_mode']],
    df['promotion_eligible'], 
) 

dept_work_contingency_per = pd.crosstab(
    [df['department'], df['work_mode']],
    df['promotion_eligible'], 
    normalize='index'
) * 100

In [ ]:
sns.heatmap(dept_work_contingency_per, annot=True, cmap='PuBu')

In [ ]:
stat, p, dof, expected = chi2_contingency(dept_work_contingency)
print(f'p-value: {p:.4f}')
print(stat)
print(dof)
print(expected)

In [ ]:
dept_work_promotion = dept_work_contingency_per.copy()
dept_work_promotion['promotion_rate'] = dept_work_promotion['Y']

In [ ]:
dept_work_promotion

In [ ]:
dept_work_promotion_plot = dept_work_promotion.reset_index()

In [ ]:
dept_work_promotion_plot

In [ ]:
sns.barplot(
    data=dept_work_promotion_plot, 
    x='department', 
    y='promotion_rate', 
    hue='work_mode'
)
plt.title('Promotion Rate by Department and Work Mode')
plt.xticks(rotation=45)
plt.ylabel('Promotion Rate (%)')
plt.tight_layout()
plt.show()

#### Analysis by experience and mode

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='work_mode', y='work_experience', hue='promotion_eligible')
plt.title('Work Experience Distribution by Work Mode and Promotion Eligibility')
plt.tight_layout()
plt.show()

In [ ]:
# Experience and promotion eligibility correlation
experience_promotion = pd.crosstab(
    pd.qcut(df['work_experience'], 4), 
    df['promotion_eligible'], 
    normalize='index'
) * 100
print("\nPromotion Rate by Experience Quartile:")
display(experience_promotion)

## Conclusions

#### Statistical Findings

A chi-square test was conducted to assess the relationship between work mode (REMOTE, HYBRID, ONSITE) and promotion eligibility:
    - Chi-square statistic: 2.3052
    - Degrees of freedom: 2
    - p-value: 0.3158

A second chi-square test was conducted to assess the relationship between work mode and department and promotion eligibility:
    - Chi-square statistic: 22.760
    - Degrees of freedom: 17
    - p-value: 0.1572

Since the p-value is greater than 0.05, we fail to reject the null hypothesis. This suggests no statistically significant association between work mode or department and promotion eligibility.